# Imports

In [4]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import glob
import tensorflow as tf
import matplotlib.pyplot as plt
from keras.models import load_model
from keras.preprocessing import image
import seaborn as sns
from sklearn import metrics

ModuleNotFoundError: No module named 'tensorflow'

# Data Prepare

In [1]:
image_path = 'C:\Users\hbkni\OneDrive\Desktop\code\static\model\real_vs_fake\real-vs-fake'

train_dir = image_path + '/train'
valid_dir = image_path + '/valid'
test_dir = image_path + '/test'

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (75898230.py, line 1)

In [ ]:
from pathlib import Path

train_path = Path(train_dir)
valid_path = Path(valid_dir)
test_path = Path(test_dir)

print("Train_dir Subfolders: ", [subfolder.name for subfolder in train_path.iterdir() if subfolder.is_dir()])
print("Valid_dir Subfolders: ", [subfolder.name for subfolder in valid_path.iterdir() if subfolder.is_dir()])
print("Test_dir Subfolders: ", [subfolder.name for subfolder in test_path.iterdir() if subfolder.is_dir()])

# Short EDA - Exploratory Data Analysis

In [ ]:
import random

In [ ]:
images_df = {
    "folder":[],
    "image_path":[],
    "label":[]
}

# Iterate over each train, valid, and test folder
for folder in ['train', 'valid', 'test']:
    # Iterate over fake and real folders (labels)
    for label in ['fake', 'real']:
        # Iterate over images with .jpg extension in the current label folder
        for img in glob.glob(f"{image_path}/{folder}/{label}/*.jpg"):
            images_df["folder"].append(folder)
            images_df["image_path"].append(img)
            images_df["label"].append(label)

In [ ]:
images_df = pd.DataFrame(images_df)

In [ ]:
images_df

Labels distribution

In [ ]:
real_grouped_df = images_df[images_df['label'] == "real"].groupby('folder')
fake_grouped_df = images_df[images_df['label'] == "fake"].groupby('folder')

In [ ]:
real_grouped_df.size()

In [ ]:
fake_grouped_df.size()

# Data Preparation

In [ ]:
image_gen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255.)

In [ ]:
train_ds = image_gen.flow_from_directory(
    train_dir,
    target_size=(256, 256),
    batch_size=64,
    class_mode='binary',
)

valid_ds = image_gen.flow_from_directory(
    valid_dir,
    target_size=(256, 256),
    batch_size=64,
    class_mode='binary'
)

test_ds = image_gen.flow_from_directory(
    test_dir,
    target_size=(256, 256),
    batch_size=64,
    class_mode='binary',
    shuffle=False)

In [ ]:
def plot_images(img, label):
    plt.figure(figsize=[12, 12])
    for i in range(16):
        plt.subplot(4, 4, i+1)
        plt.imshow(img[i])
        plt.axis('off')
        if label[i] == 0:
            plt.title("Fake")
        else:
            plt.title("Real")

# Data Visualization

In [ ]:
img,lbl = next(train_ds)
plot_images(img,lbl)

In [ ]:
for img, label in train_ds:
    print("Values: ", img[0])
    print("Label: ", label[0])
    break

In [ ]:
for img, label in train_ds:
    print(img.shape)
    print(label.shape)
    break

In [ ]:
train_ds.class_indices

# Model

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [ ]:
# Load the pre-trained ResNet50 model without the top classification layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(256, 256, 3))

# Add custom classification layers on top of the base model
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

# Create the new model
model = Model(inputs=base_model.input, outputs=output)

# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False

# Compile the model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Display the model summary
model.summary()

In [ ]:
# Train the model with data augmentation
model.fit(
    train_ds,
    epochs = 10,
    validation_data=valid_ds,
)

# Test Result

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(test_ds)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# Generate predictions for test data
predictions = model.predict(test_ds)
y_pred = (predictions > 0.5).astype(int)  # Convert probabilities to binary predictions

# Get true labels from test data generator
y_true = test_ds.classes

# Compute confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=test_ds.class_indices.keys(), 
            yticklabels=test_ds.class_indices.keys())
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# Generate classification report
class_names = list(test_ds.class_indices.keys())
report = classification_report(y_true, y_pred, target_names=class_names)
print("Classification Report:")
print(report)

In [ ]:
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# Model

In [ ]:
from tensorflow.keras.applications import DenseNet121

In [ ]:
# Load the pre-trained DenseNet121 model without the top classification layer
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(256, 256, 3))

# Add custom classification layers on top of the base model
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(256, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

# Create the new model
model = Model(inputs=base_model.input, outputs=output)

# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False

# Compile the model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Display the model summary
model.summary()

In [ ]:
# Train the model with data augmentation
model.fit(
    train_ds,
    epochs = 10,
    validation_data=valid_ds,
)

# Test Report 

In [ ]:
# Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(test_ds)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# Generate predictions for test data
predictions = model.predict(test_ds)
y_pred = (predictions > 0.5).astype(int)  # Convert probabilities to binary predictions

# Get true labels from test data generator
y_true = test_ds.classes

# Compute confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=test_ds.class_indices.keys(), 
            yticklabels=test_ds.class_indices.keys())
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# Generate classification report
class_names = list(test_ds.class_indices.keys())
report = classification_report(y_true, y_pred, target_names=class_names)
print("Classification Report:")
print(report)

In [ ]:
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# Data Augmentation

In [ ]:
train_aug = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [ ]:
train_ds_aug = train_aug.flow_from_directory(
    train_dir,
    target_size=(256, 256),
    batch_size=64,
    class_mode='binary',
)

# Train Model with data augmentation 

In [ ]:
model.fit(
    train_ds_aug,
    epochs = 10,
    validation_data=valid_ds,
)

# Result after augmentation

In [ ]:
# Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(test_ds)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

# Generate predictions for test data
predictions = model.predict(test_ds)
y_pred = (predictions > 0.5).astype(int)  # Convert probabilities to binary predictions

# Get true labels from test data generator
y_true = test_ds.classes

# Compute confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, 
            xticklabels=test_ds.class_indices.keys(), 
            yticklabels=test_ds.class_indices.keys())
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# Generate classification report
class_names = list(test_ds.class_indices.keys())
report = classification_report(y_true, y_pred, target_names=class_names)
print("Classification Report:")
print(report)

In [ ]:
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)